# **Projet 3 - Prédiction de la consommation énergétique des bâtiments de Seattle**

 * Prédire la consommation énergétique des bâtiments non résidentiels de Seattle à partir de leurs caractéristiques structurelles.
 * Réaliser une analyse exploratoire afin d'identifier les principales tendances et anomalies des données.
 * Comparer plusieurs modèles de régression supervisée pour sélectionner le plus performant.
 * Identifier les variables ayant le plus d'influence sur la consommation énergétique.*
 

## **Objectf** 
Prédire la consommation énergétique (ou les émissions) des bâtiments non résidentiels afin d'aider la ville de Seattle à atteindre la neutralité carbone en 2050.

## **ÉTAPE 4** 

### **Comparez plusieurs modèles supervisés**

#### **Importation des Libraries**

In [16]:
#importation des librairies

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Importation des librairies OK")

Importation des librairies OK


#### **Importation des Modules**

In [17]:
#Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_validate,
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

#Modèles
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

print("Importation des modules OK")

Importation des modules OK


#### **Chargement du jeu de données**

In [18]:
df = pd.read_csv("../data/processed/buildings_model.csv")

print("Importation du dataset terminée.")
print(f"Dimensions du dataset : {df.shape[0]} lignes, {df.shape[1]} colonnes")

Importation du dataset terminée.
Dimensions du dataset : 3348 lignes, 30 colonnes


### **Comparaison et Modélisation**

#### **Séparation des variables explicatives et de la cible (x et y)**

In [19]:
# Séparation des variables explicatives et de la cible

# Définition de la variable cible
target = "siteenergyuse(kbtu)"

# Séparation des variables explicatives (X) et de la cible (y)
X = df.drop(columns=[target])
y = df[target]

# Vérification des dimensions
print(f"X : {X.shape}")
print(f"y : {y.shape}")

X : (3348, 29)
y : (3348,)


> - Le jeu de données `buildings_model.csv`, sont définies à partir de ce jeu de données déjà nettoyé (étape 1) et préparé (étape 2 et 3), est chargé pour la phase de modélisation.
> - La variable cible `SiteEnergyUse(kBtu)` est séparée des variables explicatives afin de constituer les ensembles X et y.

#### **Séparation des données d'entraînement et de test**

> -  Les valeurs manquantes sont identifiées avant la séparation du jeu de données afin de préparer le prétraitement.
> -  Ces valeurs seront imputées dans le pipeline grâce à SimpleImputer, sans supprimer d'observations.

In [20]:
# Vérification des valeurs manquantes

missing_values = X.isna().sum()
missing_values = missing_values[missing_values > 0]

print(missing_values.sort_values(ascending=False))

thirdlargestpropertyusetype        2752
thirdlargestpropertyusetypegfa     2752
secondlargestpropertyusetype       1671
secondlargestpropertyusetypegfa    1671
energystarscore                     817
largestpropertyusetypegfa            11
largestpropertyusetype               11
dtype: int64


In [21]:
# Séparation des données d'entraînement et de test

from sklearn.model_selection import train_test_split

# Division des données en ensembles d'entraînement (80 %) et de test (20 %)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Vérification des dimensions des jeux de données

print(f"Train : {X_train.shape}")
print(f"Test  : {X_test.shape}")

Train : (2678, 29)
Test  : (670, 29)


> -  Le jeu de données est divisé en un ensemble d'entraînement (80 %) et un ensemble de test (20 %).
> -  Cette séparation permet d'évaluer les performances des modèles sur des données jamais vues pendant l'entraînement.

#### **Préprocessing**

> - Les variables numériques et catégorielles sont prétraitées à l'aide d'un ColumnTransformer.
> - Les valeurs manquantes sont imputées avec SimpleImputer, les variables numériques sont normalisées avec StandardScaler et les variables catégorielles sont encodées avec OneHotEncoder.

##### **Categorical Features**

##### **Preprocessor**

In [23]:
# Préprocessing des variables catégorielles et numériques

from sklearn.pipeline import Pipeline

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
("num",
 Pipeline([
     ("imputer", SimpleImputer(strategy="median")),
     ("scaler", StandardScaler())
 ]),
 numeric_features,
),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features,
        ),
    ]
)

preprocessor

C:\Users\FR103217\AppData\Local\Temp\ipykernel_23872\879553094.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

#### **DummyRegressor**

> -  Le **DummyRegressor** est utilisé comme modèle de référence (baseline).
> -  Il prédit une valeur constante correspondant à la moyenne de la variable cible du jeu d'entraînement. Les performances obtenues serviront de point de comparaison avec les autres modèles de régression.

In [24]:
# Modèle de référence

dummy_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyRegressor(strategy="mean"))
])

# Entraînement du modèle

dummy_model.fit(X_train, y_train)

# Prédictions

y_pred_dummy = dummy_model.predict(X_test)

# Calcul des métriques

dummy_r2 = r2_score(y_test, y_pred_dummy)
dummy_mae = mean_absolute_error(y_test, y_pred_dummy)
dummy_rmse = np.sqrt(mean_squared_error(y_test, y_pred_dummy))

print(f"R²   : {dummy_r2:.3f}")
print(f"MAE  : {dummy_mae:.2f}")
print(f"RMSE : {dummy_rmse:.2f}")

R²   : -0.000
MAE  : 5889953.74
RMSE : 19698255.66


> - Le **DummyRegressor** est utilisé comme modèle de référence (*baseline*). Il prédit systématiquement la moyenne de la variable cible.
> - Le **R² proche de 0** montre que ce modèle n'explique pratiquement aucune variation de la consommation énergétique.
> - Les valeurs élevées du **MAE (5 889 954)** et du **RMSE (19 698 256)** indiquent des erreurs de prédiction très importantes.

#### **Régression Linéaire**

> - Les variables catégorielles sont automatiquement encodées avec **OneHotEncoder**, tandis que les valeurs manquantes sont remplacées par un **SimpleImputer**.
> - Ce modèle permet d'évaluer la capacité d'une relation linéaire à expliquer la consommation énergétique des bâtiments.

In [25]:
# Modèle de régression linéaire

linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

# Entraînement du modèle
linear_model.fit(X_train, y_train)

# Prédictions
y_pred_lr = linear_model.predict(X_test)

# Calcul des métriques
lr_r2 = r2_score(y_test, y_pred_lr)
lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print(f"R²   : {lr_r2:.3f}")
print(f"MAE  : {lr_mae:.2f}")
print(f"RMSE : {lr_rmse:.2f}")

R²   : 0.365
MAE  : 4514223.82
RMSE : 15690182.23


> -  La régression linéaire obtient un **R² de 0,365**, ce qui signifie qu'elle explique environ **37 %** de la variabilité de la consommation énergétique.
> -  Les erreurs obtenues (**MAE = 451 424** et **RMSE = 1 569 018**) restent importantes, ce qui montre que ce modèle ne capture pas entièrement les relations entre les variables explicatives et la cible.

#### **Random Forest**

>  - Le modèle **Random Forest** est un ensemble d'arbres de décision construit selon la méthode du bagging.
>  - Il permet de capturer des relations non linéaires entre les variables explicatives et la consommation énergétique.
>  - Ce modèle est capable de modéliser des relations non linéaires et des interactions complexes entre les variables explicatives.

In [31]:
# Modèle Random Forest

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

# Entraînement du modèle
rf_model.fit(X_train, y_train)

# Prédictions
y_pred_rf = rf_model.predict(X_test)

# Calcul des métriques
rf_r2 = r2_score(y_test, y_pred_rf)
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f"R²   : {rf_r2:.3f}")
print(f"MAE  : {rf_mae:.2f}")
print(f"RMSE : {rf_rmse:.2f}")

R²   : 0.616
MAE  : 1727870.57
RMSE : 12205013.81


>  - Il obtient un coefficient de détermination de **R² = 0,616**, ce qui signifie qu'il explique environ **62 %** de la variabilité de la consommation énergétique.
>  - Les erreurs **MAE** et **RMSE** sont également plus faibles que celles obtenues avec les modèles précédents.

#### **Support Vector Regression (SVR)**

>  - Un prétraitement est appliqué afin d'imputer les valeurs manquantes et d'encoder les variables catégorielles.
>  - Ce modèle est particulièrement sensible à l'échelle des variables et peut nécessiter un temps d'entraînement plus important.

In [27]:
# Modèle Support Vector Regression

svr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", SVR())
])

# Entraînement
svr_model.fit(X_train, y_train)

# Prédictions
y_pred_svr = svr_model.predict(X_test)

# Calcul des métriques
svr_r2 = r2_score(y_test, y_pred_svr)
svr_mae = mean_absolute_error(y_test, y_pred_svr)
svr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_svr))

print(f"R²   : {svr_r2:.3f}")
print(f"MAE  : {svr_mae:.2f}")
print(f"RMSE : {svr_rmse:.2f}")

R²   : -0.040
MAE  : 4686626.58
RMSE : 20083353.36


>  - Le modèle **Support Vector Regression (SVR)** est entraîné avec les paramètres par défaut, après le prétraitement des données (imputation, normalisation et encodage).
>  - Les performances obtenues sont faibles (**R² = -0,040**), ce qui signifie que le modèle prédit moins bien que le modèle de référence.

#### **Tableau Comparatif**

>  - Les métriques utilisées sont :

>  -  **R²** : proportion de la variance expliquée par le modèle (plus il est proche de 1, meilleur est le modèle).
>  -  **MAE** : erreur absolue moyenne entre les valeurs réelles et prédites.
>  -  **RMSE** : racine de l'erreur quadratique moyenne, qui pénalise davantage les erreurs importantes.

>  - Le meilleur modèle sera retenu pour la phase d'optimisation de l'étape suivante.

In [35]:
# Tableau comparatif des performances

results = pd.DataFrame({
    "Modèle": [
        "DummyRegressor",
        "Régression Linéaire",
        "Random Forest",
        "Support Vector Regression"
    ],
    "R²": [
        dummy_r2,
        lr_r2,
        rf_r2,
        svr_r2
    ],
    "MAE": [
        dummy_mae,
        lr_mae,
        rf_mae,
        svr_mae
    ],
    "RMSE": [
        dummy_rmse,
        lr_rmse,
        rf_rmse,
        svr_rmse
    ]
})

results = results.sort_values(by="R²", ascending=False)

results["R²"] = results["R²"].map("{:.3f}".format)
results["MAE"] = results["MAE"].map(lambda x: f"{x:,.0f}".replace(",", " "))
results["RMSE"] = results["RMSE"].map(lambda x: f"{x:,.0f}".replace(",", " "))

results

,Modèle,R²,MAE,RMSE
2,Random Forest,0.616,1 727 871,12 205 014
1,Régression Linéaire,0.365,4 514 224,15 690 182
0,DummyRegressor,-0.000,5 889 954,19 698 256
3,Support Vector Regression,-0.040,4 686 627,20 083 353


>  -  Le **Random Forest** obtient les meilleures performances avec un **R² de 0,616** et les erreurs **MAE** et **RMSE** les plus faibles parmi les modèles évalués.
>  -  Ce modèle est retenu pour la phase d'optimisation de l'étape suivante, tandis que les autres modèles présentent des performances plus limitées.

### **Conclusion**

>  - Quatre modèles de régression ont été comparés afin de prédire la consommation énergétique des bâtiments.

>  Les principaux résultats sont les suivants :
>  -  Quatre modèles de régression ont été comparés afin d'évaluer leur capacité à prédire la consommation énergétique des bâtiments.
>  -  Les résultats montrent que les performances varient selon l'algorithme utilisé, le **Random Forest** obtenant les meilleurs résultats sur ce jeu de données.
>  -  Ce modèle est retenu pour la phase suivante, qui portera sur son optimisation à l'aide de **GridSearchCV** et de la validation croisée.

> L'étape suivante consistera à optimiser ce modèle par recherche d'hyperparamètres.

### **Sauvegarde des performances des modèles**

In [29]:
# Dimensions du jeu de données utilisé pour la modélisation
print(f"Nombre de lignes : {df.shape[0]}")
print(f"Nombre de colonnes : {df.shape[1]}")

Nombre de lignes : 3348
Nombre de colonnes : 30


In [30]:
from pathlib import Path

# Création du dossier
Path("../data/results").mkdir(parents=True, exist_ok=True)

# Sauvegarde du tableau comparatif
results.to_csv(
    "../data/results/model_comparison.csv",
    index=False
)

print("Tableau comparatif sauvegardé avec succès.")

Tableau comparatif sauvegardé avec succès.
